In [26]:
#Part 1: import modules and configure basic values

import re #regular expression module
import pandas as pd #pandas for data handling

#following names in ALL CAPS are settings by convention, so it looks organized
#their values are only change/defined in one place
PATH = "data/raw/ML-EdgeIIoT-dataset.csv"
LABEL, TYPE = "Attack_label","Attack_type"
SPREAD = 0.05

#NAME_PATTERN matches to column names in the dataset
NAME_PATTERN = r"(ip\.|host|time|checksum|seq|ack_raw|stream|trans_id|timestamp|payload|msg|name|uri|query|file_data|port|referer|topic)"

#These columns are artifacts, likely to hamper the learning process, and may be dropped
PROPOSED_DROP = [
    "frame.time","tcp.srcport","tcp.seq",
    "ip.src_host","ip.dst_host",
    "arp.src.proto_ipv4","arp.dst.proto_ipv4",
    "icmp.transmit_timestamp","tcp.ack_raw","tcp.checksum",
    "icmp.checksum","udp.stream","tcp.dstport","udp.port",
]

In [27]:
#Part 2: load dataset

df=pd.read_csv(PATH, low_memory = False) #loads dataset into dataframe
types = df[TYPE] #selects attack type column

'''
features are what we'll feed into the ML model
attack label is the prediction target (actual value) that the model needs to predict(is this malicious or not)
attack type is the detailed answer that the model needs to predict without being fed
it is defined whenever attack_label is 1 so it needs to be filtered out
we drop both columns from feeding

We will use attack_type for splitting and reporting later
'''

features = [c for c in df.columns if c not in (LABEL, TYPE)]
n = len(df)
print("Rows:", n, " Columns:", df.shape[1], " Features:", len(features))

Rows: 157800  Columns: 63  Features: 61


In [28]:
#value_counts() counts how many times a value occurs
#display() in Jupyter to show it neatly

display(df[LABEL].value_counts())
display(df[TYPE].value_counts())

consistent = ((df[LABEL] == 1) == (df[TYPE] != "Normal")).all()   #checks consistency of Normal traffic marking
print("Label matches type:", bool(consistent)) #check if LABEL and TYPE agree with each other

print("Majority-class share:", round(df[LABEL].value_counts(normalize=True).max(), 3))
'''
a model that always predicts positive is about 84.6% accurate
which isn't really helpful
a real model must beat this accuracy 
We use 84.6% as the baseline
'''
blocks = (types != types.shift()).sum()
print("Attack_type blocks in file order:", blocks, "of", types.nunique(), "types")

Attack_label
1    133499
0     24301
Name: count, dtype: int64

Attack_type
Normal                   24301
DDoS_UDP                 14498
DDoS_ICMP                14090
Ransomware               10925
DDoS_HTTP                10561
SQL_injection            10311
Uploading                10269
DDoS_TCP                 10247
Backdoor                 10195
Vulnerability_scanner    10076
Port_Scanning            10071
XSS                      10052
Password                  9989
MITM                      1214
Fingerprinting            1001
Name: count, dtype: int64

Label matches type: True
Majority-class share: 0.846
Attack_type blocks in file order: 15 of 15 types


In [13]:
miss = df.isna().sum()  # count empty cells per colunn
print("Columns with missing values:", list(miss[miss>0].index))  # keep only coluns with at least one empty cell

nunique = df.nunique(dropna=False) #count distinct values per column
constant = [c for c in features if nunique[c] <= 1]
print("Constant columns:", constant)

#models need numbers to work with
#text columns need to be dropped

text_cols = [c for c in features if not pd.api.types.is_numeric_dtype(df[c])]
print("Text columns:", text_cols)

Columns with missing values: []
Constant columns: ['icmp.unused', 'http.tls_port', 'dns.qry.type', 'dns.retransmit_request_in', 'mqtt.msg_decoded_as', 'mbtcp.len', 'mbtcp.trans_id', 'mbtcp.unit_id']
Text columns: ['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.src.proto_ipv4', 'http.file_data', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'tcp.options', 'tcp.payload', 'tcp.srcport', 'dns.qry.name.len', 'mqtt.conack.flags', 'mqtt.msg', 'mqtt.protoname', 'mqtt.topic']


In [25]:
#rows that are exactly identical will inflate scores
#cannot be separated by any model and will limit accuracy

h = pd.util.hash_pandas_object(df[features], index = False) #fingerprint each row
dups = h.duplicated() #check for duplicates
print("Repeated rows:", dups.sum())
print(df.loc[dups, TYPE].value_counts())

conflict = (df.groupby(h.values)[LABEL].nunique() > 1).sum()
print("Feature patterns with BOTH labels:", conflict)

Repeated rows: 814
Attack_type
MITM    814
Name: count, dtype: int64
Feature patterns with BOTH labels: 0


In [16]:
#record artifacts
#these are columns that are recorded differently in some attack types
#ML model could use these as shortcuts during learning
#could still be used, need to be assessed properly
#their kind of value differs between attack types

#text columns and numeric columns are both present

rows = []
for c in features:
    if pd.api.types.is_numeric_dtype(df[c]): #catch numeric types here
        bad = df[c].notna() & (df[c] % 1 != 0)
        kind = "non-integer"
    else:
        bad = pd.to_numeric(df[c], errors="coerce").isna() #catch non numeric types and convert to numeric if possible?
        kind = "non-numeric"
    share = bad.groupby(types).mean() #group rows by attack type, calculate share of odd values
    spread = share.max() - share.min() 
    #spread ~ 1 : Some attacks are odd and others aren't for this column
    #spread ~ 0 : Every attack type looks the same for this column
    if spread > SPREAD: #keep only those columns whose spreads are more than 5%
        rows.append((c, kind, round(spread, 3), share.idxmax(), share.idxmin()))

scan = pd.DataFrame(rows, columns = ["column", "kind", "spread","highest_type","lowest_type"])
display(scan.sort_values("spread", ascending=False))

,column,kind,spread,highest_type,lowest_type
0,frame.time,non-numeric,1.000,Backdoor,MITM
2,ip.dst_host,non-numeric,1.000,DDoS_ICMP,MITM
10,tcp.options,non-numeric,0.995,Vulnerability_scanner,DDoS_ICMP
11,tcp.payload,non-numeric,0.855,Vulnerability_scanner,DDoS_ICMP
5,http.file_data,non-numeric,0.783,Vulnerability_scanner,Backdoor
1,ip.src_host,non-numeric,0.719,DDoS_ICMP,MITM
9,http.request.version,non-numeric,0.428,Vulnerability_scanner,Backdoor
7,http.request.method,non-numeric,0.428,Vulnerability_scanner,Backdoor
8,http.request.full_uri,non-numeric,0.426,Vulnerability_scanner,Backdoor
13,tcp.srcport,non-numeric,0.302,MITM,Backdoor


In [17]:
#for each malicious column, does a single value from a column tend to point to a single label?
#model could end up memorising the value instead of learning the attack behaviour

for c in features: 
    if re.search(NAME_PATTERN, c) or nunique[c] > 0.5 * n : #search columns that have sus words or ones that contain distinct values in more than half the rows
        nu = nunique[c] #number of distinct values
        line = f"{c:30s} unique={nu}" 
        if 1 < nu <= 50000: #checks for columns with more than one distinct value
            g = df.groupby(c, dropna=False)[LABEL].nunique() #group rows by distinct value of each column
            share = df[c].isin(g.index[g == 1]).mean()  
            line += f" only-one-class share = {share:.1%}"
        print(line)

frame.time                     unique=155186
ip.src_host                    unique=19090 only-one-class share = 42.4%
ip.dst_host                    unique=8084 only-one-class share = 32.8%
icmp.checksum                  unique=13187 only-one-class share = 9.3%
icmp.seq_le                    unique=13824 only-one-class share = 9.9%
icmp.transmit_timestamp        unique=84 only-one-class share = 0.1%
http.file_data                 unique=496 only-one-class share = 25.8%
http.request.uri.query         unique=1665 only-one-class share = 12.9%
http.referer                   unique=4 only-one-class share = 19.4%
http.request.full_uri          unique=4073 only-one-class share = 38.8%
http.tls_port                  unique=1
tcp.ack_raw                    unique=94716
tcp.checksum                   unique=55513
tcp.dstport                    unique=23188 only-one-class share = 62.6%
tcp.payload                    unique=27369 only-one-class share = 52.5%
tcp.seq                        unique=1

In [21]:
drop = [c for c in PROPOSED_DROP if c in df.columns] #columns proposed to drop
rest = [c for c in features if c not in drop and c not in constant] # Columns left if proposed ones are removed
num_cols = [c for c in rest if pd.api.types.is_numeric_dtype(df[c])] # split rest into numeric and text columns
txt_cols = [c for c in rest if c not in num_cols]

def distinct_by_type(cols):
    k = pd.util.hash_pandas_object(df[cols + [TYPE]], index=False) 
    return(~k.duplicated()).groupby(types).sum()#counts distinct values per attack type

preview = pd.DataFrame({
    "raw": types.value_counts(), # row count per type
    "numeric only": distinct_by_type(num_cols), #distinct rows left if models use only numeric columns
    "numeric + text": distinct_by_type(num_cols + txt_cols), #distinct rows if text columns are also included
})

display(preview)
print(preview.sum())

,raw,numeric only,numeric + text
Attack_type,,,
Backdoor,10195,4598,9506
DDoS_HTTP,10561,1399,9412
DDoS_ICMP,14090,12694,12694
DDoS_TCP,10247,6010,6010
DDoS_UDP,14498,1403,1403
Fingerprinting,1001,6,6
MITM,1214,116,154
Normal,24301,4042,4284
Password,9989,42,9933


raw               157800
numeric only       46972
numeric + text    104662
dtype: int64
